# 07 — Chennai: zero-shot domain transfer

Applies the OSCD-trained model, **unchanged**, to a bi-temporal Sentinel-2 pair over Chennai
acquired independently via Google Earth Engine — no Chennai labels, no fine-tuning.

Deliverables:
1. a reusable Sentinel-2 L2A ingestion path for an arbitrary AOI (GEE acquisition + lazy GeoTIFF reader),
2. a georeferenced change-probability map for the full Chennai metro + fringe,
3. a qualitative generalization analysis (city map + hotspot panels) with a full scene record.

OSCD-derived labels exist only for the 24 OSCD cities, so the Chennai output is evaluated
qualitatively — consistent with a zero-shot transfer study; OSM-derived labels are future work.

**One-time setup (outside Colab):** register a free noncommercial Earth Engine project at
https://code.earthengine.google.com/register and note its project ID (needed in the auth cell below).

**Storage contract** — code lives on GitHub (branch `chennai`), data & checkpoints live on Drive
(`MyDrive/s2cd`), `/content` is disposable scratch. Run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # Chennai phase lives on this branch
if os.path.exists(REPO):
    !git -C {REPO} fetch origin
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]
# tripwire: s2_l2a.py exists only on the chennai branch — a failed checkout
# (branch not pushed yet?) must not be silently worked around
assert os.path.exists(os.path.join(REPO, "src/s2cd/data/s2_l2a.py")), \
    f"s2_l2a.py missing — is branch '{BRANCH}' pushed to origin?"
print("chennai branch checked out")


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device

device = get_device()
assert device.type == "cuda", "Runtime > Change runtime type > select a GPU"
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


## Acquire the bi-temporal pair from Earth Engine

Both dates use the same post-monsoon window (Jan–Mar): clearest skies over Chennai, and
same-season pairing suppresses spurious 'change' from vegetation phenology. Scenes are ranked
by `CLOUDY_PIXEL_PERCENTAGE` and the least-cloudy one is exported with the SAME 10 bands the
OSCD pipeline uses — native 10 m bands plus the 20 m bands bilinearly resampled at export time,
mirroring `oscd.py`'s preprocessing — as single uint16 GeoTIFFs to Drive.

In [ ]:
!pip install -q earthengine-api
import ee
ee.Authenticate()   # opens a browser tab for the one-time OAuth consent

EE_PROJECT = "your-gee-project-id"  # TODO: from code.earthengine.google.com/register
assert EE_PROJECT != "your-gee-project-id", "set EE_PROJECT to your registered GEE project id"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized:", EE_PROJECT)


In [ ]:
from s2cd.data.gee_acquire import CHENNAI_RECT, find_scene, list_scenes

T0_PERIOD = ("2018-01-01", "2018-03-31")   # post-monsoon: clearest window
T1_PERIOD = ("2025-01-01", "2025-03-31")
CLOUD_MAX = 5.0

print("AOI [W, S, E, N]:", CHENNAI_RECT)
for label, (d0, d1) in [("t0", T0_PERIOD), ("t1", T1_PERIOD)]:
    print(f"\ncandidates for {label} ({d0}..{d1}):")
    for s in list_scenes(CHENNAI_RECT, d0, d1, cloud_max=CLOUD_MAX, limit=5):
        print(f"  {s['date']}  cloud {s['cloud_pct']:4.1f}%  {s['id']}")

scene_t0 = find_scene(CHENNAI_RECT, *T0_PERIOD, cloud_max=CLOUD_MAX)
scene_t1 = find_scene(CHENNAI_RECT, *T1_PERIOD, cloud_max=CLOUD_MAX)
print(f"\nselected t0: {scene_t0['date']} (cloud {scene_t0['cloud_pct']}%)")
print(f"selected t1: {scene_t1['date']} (cloud {scene_t1['cloud_pct']}%)")


In [ ]:
from s2cd.data.gee_acquire import export_scene, wait_for_tasks

# ~1.2 GB per date as uint16 -> single-file Drive exports (EE would split
# larger exports into chunks; load_scene stitches those if the AOI grows)
tasks = [export_scene(scene_t0["image"], CHENNAI_RECT, "chennai_t0"),
         export_scene(scene_t1["image"], CHENNAI_RECT, "chennai_t1")]
wait_for_tasks(tasks)   # polls every 30 s; typically 5-15 min per export


## Stage & lazy-load the scenes

`L2AScene` reads only the window `predict_city` asks for — the full 2-date x 10-band stack
never materializes in RAM. It duck-types the `read_city` ndarray contract (`.shape` +
window slices), so `evaluate.predict_city` runs unchanged.

In [ ]:
import shutil
from pathlib import Path
from s2cd.data import load_scene

DRIVE_CH = store.drive / "chennai"       # export folder was s2cd/chennai
stage = Path("/content/chennai"); stage.mkdir(exist_ok=True)
for name in ["chennai_t0", "chennai_t1"]:
    src = DRIVE_CH / f"{name}.tif"
    assert src.exists(), f"{src} missing — did the export cell complete?"
    shutil.copy(src, stage / f"{name}.tif")

t0 = load_scene(stage / "chennai_t0.tif")
t1 = load_scene(stage / "chennai_t1.tif")
assert t0.shape == t1.shape, f"scene shapes differ: {t0.shape} vs {t1.shape}"
print(type(t0).__name__, t0.shape, t0.crs)


## Zero-shot inference over the full AOI

The model checkpoint is the OSCD-trained `exp1` best — nothing is retrained. Windows stream
from disk (chip 256, stride 256, probability-averaged like the OSCD test maps).

In [ ]:
import time
from s2cd.data.dataset import load_stats_from_manifest
from s2cd.evaluate import load_best_model, predict_city

model, cfg, device = load_best_model("configs/train.yaml", store)
stats = load_stats_from_manifest(store.read_manifest())

t_infer = time.time()
prob = predict_city(model, t0, t1, stats, chip=256, stride=256)
infer_s = time.time() - t_infer
print(f"{t0.shape[1]}x{t0.shape[2]} px in {infer_s:.1f}s on {describe_device(device)}")


## City change map

Same conventions as the OSCD report maps: RGB stretched to 0-0.3 reflectance,
P(change) on the magma sequential ramp pinned to the full 0-1 range (a colorbar carries
the scale; probabilities are never read by hue alone).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from s2cd.utils import load_config

bands = load_config("configs/data.yaml")["bands"]
rgb = [bands.index("B04"), bands.index("B03"), bands.index("B02")]
h, w = prob.shape
qh, qw = h // 8, w // 8               # decimated quicklooks for display

# a[rgb] (NOT a[[rgb]]): rgb is a list, so double brackets nest it as a
# 2-D index array -> (1, 3, H, W) and the transpose below fails.
stretch = lambda a: np.clip(a[rgb] / 0.3, 0, 1).transpose(1, 2, 0)
q0 = t0.read(out_shape=(qh, qw))
q1 = t1.read(out_shape=(qh, qw))

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
axes[0].imshow(stretch(q0)); axes[0].set_title(f"t0  {scene_t0['date']} (cloud {scene_t0['cloud_pct']}%)")
axes[1].imshow(stretch(q1)); axes[1].set_title(f"t1  {scene_t1['date']} (cloud {scene_t1['cloud_pct']}%)")
im = axes[2].imshow(prob, cmap="magma", vmin=0, vmax=1)
axes[2].set_title("P(change)  zero-shot transfer")
fig.colorbar(im, ax=axes[2], fraction=0.04, pad=0.02, label="P(change)")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
fig.savefig(rep / "map_chennai.png", dpi=150, bbox_inches="tight")
store.push_report("map_chennai.png")
plt.show()


## Hotspot zoom panels

The city map shows where probability concentrates; these panels zoom the top-3 512 px cells
(non-overlapping grid, ranked by mean probability) so real construction/redevelopment can be
distinguished from residual cloud or sensor differences — each row is one location: t0, t1, P(change).

In [ ]:
K, win = 3, 512
cells = [(prob[y:y+win, x:x+win].mean(), y, x)
         for y in range(0, h - win + 1, win)
         for x in range(0, w - win + 1, win)]
top = sorted(cells, reverse=True)[:K]

fig, axes = plt.subplots(K, 3, figsize=(13, 4.4 * K))
for i, (mean_p, y, x) in enumerate(top):
    axes[i, 0].imshow(stretch(t0[:, y:y+win, x:x+win]))
    axes[i, 0].set_ylabel(f"#{i+1}", rotation=0, labelpad=18)
    axes[i, 1].imshow(stretch(t1[:, y:y+win, x:x+win]))
    axes[i, 2].imshow(prob[y:y+win, x:x+win], cmap="magma", vmin=0, vmax=1)
    axes[i, 2].set_title(f"P(change) mean={mean_p:.2f}", fontsize=10)
    print(f"hotspot #{i+1}: mean P={mean_p:.3f} "
          f"@ ({x*10//1000} km, {y*10//1000} km from NW corner)")
axes[0, 0].set_title(f"t0  {scene_t0['date']}")
axes[0, 1].set_title(f"t1  {scene_t1['date']}")
for ax in axes.flat:
    ax.set_axis_off()
plt.tight_layout()
fig.savefig(rep / "map_chennai_hotspots.png", dpi=150, bbox_inches="tight")
store.push_report("map_chennai_hotspots.png")
plt.show()


## Export artifacts

The probability map is written as a **georeferenced GeoTIFF** (same CRS/transform as the
source scene, so it opens in QGIS/Google Earth Pro), plus a machine-readable scene record —
enough to reproduce every number in the analysis. Both land in `MyDrive/s2cd/reports/`.

In [ ]:
import json
import rasterio

profile = t0.profile               # georeferencing from the source scene
profile.update(driver="GTiff", count=1, dtype="float32", compress="deflate")
with rasterio.open(rep / "change_prob_chennai.tif", "w", **profile) as dst:
    dst.write(prob, 1)
store.push_report("change_prob_chennai.tif")

record = {
    "scope": "zero-shot transfer: OSCD-trained model applied unchanged; no Chennai labels",
    "model_run": cfg["run_name"],
    "aoi_wsen": CHENNAI_RECT,
    "bands": bands,
    "chip": 256, "stride": 256,
    "scenes": {"t0": {k: scene_t0[k] for k in ("id", "date", "cloud_pct")},
                "t1": {k: scene_t1[k] for k in ("id", "date", "cloud_pct")}},
    "prob_shape": [int(h), int(w)],
    "device": describe_device(device),
    "inference_seconds": round(infer_s, 1),
}
with open(rep / "chennai_results.json", "w") as f:
    json.dump(record, f, indent=1)
store.push_report("chennai_results.json")
print("pushed: map_chennai.png, map_chennai_hotspots.png, "
      "change_prob_chennai.tif, chennai_results.json -> Drive")


## Interpretation notes

- **What counts as signal:** spatially coherent high-probability clusters that line up with
visible t0-to-t1 construction (bare land to built-up, or demolition) in the RGB pair.
- **What to discount:** residual cloud/shadow in either date — same-season pairing keeps
vegetation phenology out, but clouds are per-scene.
- **No ground truth here:** Chennai has no labeled change dataset, so this is a qualitative
generalization result, not a scored benchmark — the scored benchmark stays on OSCD test cities.
- **Reproducibility:** scene IDs, dates, cloud %, AOI and timing all live in `chennai_results.json`;
re-run with a different AOI by changing `CHENNAI_RECT` and re-running the export cells.